# 5. Model Training

The three classification models used in the main notebook are trained on the prepared data.

In [1]:

import pandas as pd
import numpy as np
import joblib

processed_data = joblib.load("../data/03_processed_data.pkl")

X_train = processed_data["X_train"]
X_test = processed_data["X_test"]
y_train = processed_data["y_train"]
y_test = processed_data["y_test"]
X_train_scaled = processed_data["X_train_scaled"]
X_test_scaled = processed_data["X_test_scaled"]

print("Processed data loaded successfully!")
print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)


Processed data loaded successfully!
Training features: (19760, 21)
Testing features: (4940, 21)


In [2]:
print("Variables containing 'model':")

for name in list(globals().keys()):
    if "model" in name.lower():
        print(name)

Variables containing 'model':


## Logistic Regression Model

In [3]:
print("X_train shape:", X_train.shape)
print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train shape:", y_train.shape)

print("\nMissing values in X_train_scaled:",
      np.isnan(X_train_scaled).sum())

print("Missing values in y_train:",
      y_train.isna().sum())

print("\nX_train_scaled type:", type(X_train_scaled))
print("y_train type:", type(y_train))

X_train shape: (19760, 21)
X_train_scaled shape: (19760, 21)
y_train shape: (19760,)

Missing values in X_train_scaled: 0
Missing values in y_train: 0

X_train_scaled type: <class 'numpy.ndarray'>
y_train type: <class 'pandas.core.series.Series'>


In [4]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_scaled, y_train)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


## Decision Tree Model

A Decision Tree is a supervised learning algorithm that makes predictions
by splitting the data based on feature values.

It is easy to interpret and can capture non-linear relationships between
customer characteristics and loan recovery.

In [5]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=6
)

decision_tree_model.fit(X_train, y_train)

print("Decision Tree model trained successfully!")

Decision Tree model trained successfully!


## Random Forest Model

Random Forest is an ensemble learning algorithm that combines multiple
Decision Trees to make a more reliable prediction.

It generally provides better performance and reduces overfitting compared
with using a single Decision Tree.

In [6]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=8
)

random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


### Saving the trained models

In [7]:

trained_models = {
    "logistic_model": logistic_model,
    "decision_tree_model": decision_tree_model,
    "random_forest_model": random_forest_model
}

joblib.dump(trained_models, "../data/05_trained_models.pkl")
print("All trained models saved successfully!")


All trained models saved successfully!


In [8]:
print("Missing values in X_train:")
print(X_train.isna().sum().sort_values(ascending=False).head(15))

Missing values in X_train:
age                         0
income                      0
credit_score                0
loan_amount                 0
loan_term_months            0
interest_rate               0
debt_to_income              0
existing_loans              0
previous_defaults           0
payment_delays              0
employment_years            0
collateral_value            0
contact_success_rate        0
employment_type_Contract    0
employment_type_Salaried    0
dtype: int64


In [9]:
print("Available trained models:")
print(trained_models.keys())

Available trained models:
dict_keys(['logistic_model', 'decision_tree_model', 'random_forest_model'])


## Recall and ROC-AUC Optimization

Recall and ROC-AUC are the primary evaluation metrics for this project.

Recall measures the proportion of actual recovered loans that are correctly
identified by the model. A high recall is useful because missing a potentially
recoverable loan may result in a missed recovery opportunity.

ROC-AUC measures how effectively the model distinguishes between recovered
and non-recovered loans across different classification thresholds.

The models are first evaluated using the standard probability threshold of
0.50. Threshold analysis is then performed on the final Random Forest model
to understand the trade-off between recall and precision.

In [10]:
import numpy as np
from sklearn.metrics import recall_score, precision_score, roc_auc_score

# Get recovery probabilities from Random Forest
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

# ROC-AUC
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest ROC-AUC:", round(roc_auc_rf, 4))

print("\nThreshold Analysis")
print("------------------")
print("Threshold | Recall | Precision")

thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

for threshold in thresholds:

    y_pred_threshold = (y_prob_rf >= threshold).astype(int)

    recall = recall_score(y_test, y_pred_threshold)
    precision = precision_score(y_test, y_pred_threshold)

    print(
        threshold,
        "|",
        round(recall, 4),
        "|",
        round(precision, 4)
    )

Random Forest ROC-AUC: 0.9134

Threshold Analysis
------------------
Threshold | Recall | Precision
0.3 | 0.9751 | 0.7796
0.4 | 0.9511 | 0.814
0.5 | 0.9184 | 0.8467
0.6 | 0.8729 | 0.8814
0.7 | 0.7932 | 0.9155


### Threshold Selection

The threshold analysis demonstrates that changing the probability threshold
changes the balance between recall and precision.

For this project, the standard threshold of 0.50 is retained because it
provides a strong balance between identifying recovered loans and avoiding
excessive false positives.

The Random Forest model is selected as the final model because it provides
the highest recall among the three evaluated models while maintaining a
strong ROC-AUC.

In a real-world deployment, the threshold could be adjusted based on
collection costs, available recovery resources, and the business impact
of missed recovery opportunities.

In [11]:
import joblib
import os

model_path = "../data/05_trained_models.pkl"

print("Model file exists:", os.path.exists(model_path))

trained_models = joblib.load(model_path)

print("Saved models:", list(trained_models.keys()))

Model file exists: True
Saved models: ['logistic_model', 'decision_tree_model', 'random_forest_model']
